# Experimentos CNN vs ResNet50
Este notebook executa os experimentos em **MNIST** e **CIFAR10** usando:
- CNN customizada com regularização (BatchNorm, Dropout e Weight Decay)
- ResNet50 pré-treinada (fine-tuning da camada final)

In [ ]:
# !pip install -r ../requirements.txt
import os
os.chdir('..')
print('Diretório atual:', os.getcwd())

In [ ]:
!python scripts/run_experiments.py --epochs 5 --batch-size 64 --output-dir results

In [ ]:
!python scripts/plot_results.py --results-dir results --output-dir results/plots

## Visualização das ativações dos kernels
A célula abaixo captura os mapas de ativação da primeira camada convolucional da CNN customizada.

In [ ]:
import math
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms
from src.cnn_model import CustomCNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CustomCNN(in_channels=1, num_classes=10, dropout_rate=0.4).to(device)
model.eval()

sample = datasets.MNIST(
    root='data', train=False, download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
)[0][0].unsqueeze(0).to(device)

with torch.no_grad():
    activations = model.features[:4](sample).squeeze(0).cpu()

num_maps = min(16, activations.shape[0])
cols = 4
rows = math.ceil(num_maps / cols)
fig, axes = plt.subplots(rows, cols, figsize=(10, 8))
axes = axes.flatten()

for i in range(num_maps):
    axes[i].imshow(activations[i], cmap='viridis')
    axes[i].set_title(f'Kernel {i+1}')
    axes[i].axis('off')

for i in range(num_maps, len(axes)):
    axes[i].axis('off')

plt.suptitle('Ativações da 1ª camada convolucional (MNIST)')
plt.tight_layout()
plt.show()